# 01 — 主解析: RF基準とGaussian Processカーネル比較

このノートブックが本プロジェクトの中心です。受領した3つのCSVを使い、(1) R `randomForest` + residual PLS5の再現、(2) 同じ固定10-foldで複数のGPRカーネルを比較、(3) 最良GPRがRFをどこで改善し、どこで失敗するかを確認します。

重要: 標準化、角度変換、分散ゼロ列除去、`X_proc` のPCAは各訓練foldだけでfitします。

In [ ]:
# GitHubから開いたColabはファイルを自動取得しないため、最初にcloneします。
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/futoshi-futami/Chemistory.git'
REPO_REF = os.environ.get('CHEMISTORY_REF', 'main')
FALLBACK_REF = 'agent/rbf-kernel-comparison'  # PR #2。mainへmerge後はfallback不要
cwd = Path.cwd()
if (cwd / 'pyproject.toml').exists():
    PROJECT_ROOT = cwd
elif (Path('/content/Chemistory') / 'pyproject.toml').exists():
    PROJECT_ROOT = Path('/content/Chemistory')
elif 'google.colab' in sys.modules:
    PROJECT_ROOT = Path('/content/Chemistory')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    raise FileNotFoundError('Chemistory project rootでノートブックを実行してください。')
# PRのレビュー中も実行可能にし、merge後は自動的にmainを使います。
if 'google.colab' in sys.modules and not (PROJECT_ROOT / 'src/chemistory_gpr/kernels.py').exists():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--depth', '1', 'origin', FALLBACK_REF], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, 'scripts/prepare_data.py'], check=True)
# このセルだけで、後続セルのchemistory_gpr importまで準備します。
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
src_dir = str(PROJECT_ROOT / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
import importlib
importlib.invalidate_caches()
import chemistory_gpr
print('PROJECT_ROOT =', PROJECT_ROOT)
print('chemistory_gpr =', chemistory_gpr.__file__)

In [ ]:
# Python環境
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
RESULTS = PROJECT_ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

## A. 入力データの整合性検査

In [ ]:
from chemistory_gpr.handoff import load_handoff_data

DATA_DIR = PROJECT_ROOT / 'data' / 'gpr_handoff'
data = load_handoff_data(DATA_DIR)
print('n =', len(data.y))
print('base feature count =', data.base.shape[1] - 2)
print('X_proc feature count =', data.xproc.shape[1] - 1)
print('fold counts =', pd.Series(data.fold_id).value_counts().sort_index().to_dict())
display(pd.read_csv(DATA_DIR / '04_reference_RF_results.csv'))

## B. R randomForestの再現

RFはRの `randomForest` と `pls::plsr(method='simpls')` をそのまま使います。R 3.6で `sample()` の方式が変わったため、現在の `Rejection` と旧 `Rounding` を両方試します。

In [ ]:
import shutil

IN_COLAB = 'google.colab' in sys.modules
if shutil.which('Rscript') is None and IN_COLAB:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'r-base', 'r-cran-randomforest', 'r-cran-pls'], check=True)
elif shutil.which('Rscript') is None:
    print('この環境にはRscriptがないためRFセルだけをスキップします。Colabでは自動導入されます。')
else:
    package_check = 'missing <- c("randomForest","pls")[!sapply(c("randomForest","pls"), requireNamespace, quietly=TRUE)]; if(length(missing)) install.packages(missing, repos="https://cloud.r-project.org")'
    subprocess.run(['Rscript', '-e', package_check], check=True)
print('Rscript =', shutil.which('Rscript'))

In [ ]:
if shutil.which('Rscript'):
    subprocess.run([sys.executable, 'scripts/run_rf_reproduction.py'], check=True)
    rf_metrics = pd.read_csv(RESULTS / 'rf_reproduction_all_metrics.csv')
    rf_comparison = pd.read_csv(RESULTS / 'rf_reproduction_comparison.csv')
    display(rf_metrics)
    display(rf_comparison)
else:
    print('RF再現は未実行です。')

## C. GPR候補の固定10-fold比較

周期角度 + `X_proc` fold内PCA8を共通にし、Matérn 1/2・3/2・5/2、等方RBF、Rational Quadratic、RBF-ARD、Linearを比較します。すべてWhiteKernel付きです。

既定では計算の重い120次元RBF-ARDだけコミット済み結果を読み、他6候補を再fitします。`INCLUDE_ARD=True` でARDも再計算できます。`ARD_RESTARTS=0` でも帯域幅は第二種最尤法で最適化されます。`R2` は相関係数の二乗ではなく、`1-SSE/SST` です。

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from chemistory_gpr.handoff import cross_validate_handoff, handoff_kernel_candidates

INCLUDE_ARD = False  # TrueではColabで数分以上かかることがあります
ARD_RESTARTS = 0
committed_metrics = pd.read_csv(RESULTS / 'gpr_handoff_metrics.csv')
candidate_metrics = []
candidate_predictions = {}
configs = handoff_kernel_candidates(rbf_ard_restarts=ARD_RESTARTS)
if not INCLUDE_ARD:
    configs = [config for config in configs if not config.ard]
warnings.filterwarnings('ignore', category=ConvergenceWarning)
for config in configs:
    print('Running', config.name)
    prediction, metrics = cross_validate_handoff(data, config)
    candidate_predictions[config.name] = prediction
    metrics = {k: v for k, v in metrics.items() if k not in {'kernels','kernel_diagnostics'}}
    candidate_metrics.append(metrics)
    prediction.to_csv(RESULTS / f'gpr_handoff_oof_{config.name}.csv', index=False)
metric_table = pd.DataFrame(candidate_metrics)
if not INCLUDE_ARD:
    ard_reference = committed_metrics.loc[committed_metrics['ard'].astype(bool)].copy()
    metric_table = pd.concat([metric_table, ard_reference], ignore_index=True)
    for name in ard_reference['model']:
        candidate_predictions[name] = pd.read_csv(RESULTS / f'gpr_handoff_oof_{name}.csv')
metric_table = metric_table.sort_values('R2', ascending=False)
metric_table.to_csv(RESULTS / 'gpr_handoff_metrics.csv', index=False)
metric_table['upper_bound_fraction'] = np.where(
    metric_table['length_scales_total'] > 0,
    metric_table['length_scales_at_upper_bound_total'] / metric_table['length_scales_total'],
    np.nan,
)
display(metric_table[['model','kernel_family','ard','optimizer_restarts','R2','corr2','RMSE','MAE','coverage_95','NLPD','upper_bound_fraction']])

In [ ]:
# RFを主基準に加え、fold別・試料別の挙動診断を作成
from chemistory_gpr.handoff_report import build_handoff_report

report_paths = build_handoff_report(DATA_DIR, RESULTS)
primary = pd.read_csv(report_paths['comparison'])
behavior = pd.read_csv(report_paths['behavior_summary'])
fold_vs_rf = pd.read_csv(report_paths['best_vs_rf_fold_metrics'])
largest_errors = pd.read_csv(report_paths['largest_errors'])
display(primary[['rank_R2','source','model','kernel_family','R2','RMSE','MAE','coverage_95','NLPD','fold_RMSE_wins_out_of_10']])
display(behavior)
display(largest_errors[['file_key','fold','y','pred_mean','pred_std','rf_pred','gpr_abs_error','rf_abs_error']].head(10))

In [ ]:
# 最良GPRとRFのOOF予測・fold別RMSE・不確実性の挙動
best_name = metric_table.iloc[0]['model']
best = candidate_predictions[best_name]
paired = pd.read_csv(report_paths['paired_predictions'])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(paired['y'], paired['rf_pred'], s=18, alpha=.55, label='RF')
axes[0].scatter(paired['y'], paired['pred_mean'], s=18, alpha=.55, label='best GPR')
limits = [paired['y'].min(), paired['y'].max()]
axes[0].plot(limits, limits, '--', color='black')
axes[0].set(xlabel='Observed y', ylabel='OOF prediction', title='RF vs best GPR')
axes[0].legend()
x = np.arange(len(fold_vs_rf))
axes[1].bar(x-.2, fold_vs_rf['RF_RMSE'], width=.4, label='RF')
axes[1].bar(x+.2, fold_vs_rf['RMSE'], width=.4, label='best GPR')
axes[1].set(xticks=x, xticklabels=fold_vs_rf['fold'], xlabel='fold', ylabel='RMSE', title='Performance is fold-dependent')
axes[1].legend()
axes[2].scatter(paired['pred_std'], paired['gpr_abs_error'], s=20, alpha=.65)
rho = paired['pred_std'].corr(paired['gpr_abs_error'], method='spearman')
axes[2].set(xlabel='GPR predictive std', ylabel='absolute error', title=f'Uncertainty ranking: Spearman={rho:.3f}')
plt.tight_layout(); plt.show()

## D. 分子軸の角度帯ごとの挙動

C3→H3とC6→H6は全例でほぼ反対向きなので、2本を別々にbin分けせず、C3→H3を反転してC6→H6と円周平均した `axis_angle_deg` を主解析に使います。これは化学結合角ではなく、共通xy座標系における分子軸の方位です。角度帯は結果を見た後の探索的層別であり、各帯の勝者を確定モデルとするにはnested CVが必要です。

In [ ]:
from chemistory_gpr.angle_report import build_handoff_angle_report

angle_paths = build_handoff_angle_report(DATA_DIR, RESULTS)
angle_summary = pd.read_csv(angle_paths['behavior_summary'])
axis_data_summary = pd.read_csv(angle_paths['axis_data_summary'])
angle_winners = pd.read_csv(angle_paths['winners'])
angle_metrics = pd.read_csv(angle_paths['method_metrics'])
axis_associations = pd.read_csv(angle_paths['axis_feature_associations'])
high_angle_contrasts = pd.read_csv(angle_paths['high_angle_structural_contrasts'])
series_summary = pd.read_csv(angle_paths['series_summary'])
axis_winners = angle_winners.loc[angle_winners['angle_view'].eq('molecular_axis')].copy()
bin_order = ['[-30,-10)', '[-10,10)', '[10,30)', '[30,50)', '[50,70]']
axis_winners['angle_bin'] = pd.Categorical(axis_winners['angle_bin'], bin_order, ordered=True)
axis_winners = axis_winners.sort_values('angle_bin')
display(angle_summary)
display(axis_data_summary)
display(axis_winners[['angle_bin','n','y_mean','y_std','best_overall_model','best_overall_RMSE','best_GPR_model','best_GPR_RMSE','RF_RMSE']])
print('分子軸と最も強く対応する非角度特徴（探索的）')
display(axis_associations.head(10))
print('50–70°の低応答9例と残り32例の構造差（探索的）')
display(high_angle_contrasts.head(10))
print('高角度・低応答枝を含むfile_key prefix候補系列')
display(series_summary.loc[series_summary['n_high_angle_y_below_30'].gt(0)])

In [ ]:
# 応答の角度依存をH3近傍環境・面外傾きで色分けし、右に角度帯別RMSEを表示
angle_features = pd.read_csv(angle_paths['angle_features'])
base_for_plot = pd.read_csv(DATA_DIR / '01_base_summary_first_angle.csv')
angle_plot = angle_features.merge(base_for_plot[['file_key','O_H3_count_d5']], on='file_key', validate='one_to_one')
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
scatter = axes[0].scatter(angle_plot['axis_angle_deg'], angle_plot['y'], c=angle_plot['O_H3_count_d5'], cmap='viridis', s=38, alpha=.8)
axes[0].axvspan(50, 70, color='tab:red', alpha=.08)
axes[0].set(xlabel='Molecular-axis azimuth (degree)', ylabel='Observed y', title='High-angle branch and local H3 environment')
fig.colorbar(scatter, ax=axes[0], label='O_H3_count_d5')
tilt = axes[1].scatter(angle_plot['axis_angle_deg'], angle_plot['y'], c=angle_plot['axis_abs_elevation_deg_proxy'], cmap='plasma', s=38, alpha=.8)
axes[1].axvspan(50, 70, color='tab:red', alpha=.08)
axes[1].set(xlabel='Molecular-axis azimuth (degree)', ylabel='Observed y', title='Absolute out-of-plane tilt proxy')
fig.colorbar(tilt, ax=axes[1], label='absolute elevation proxy (degree)')
model_labels = {
    'RF_current_residualPLS5': 'RF',
    'base_cyclic_xproc_pca8_matern12': 'Matérn 1/2',
    'base_cyclic_xproc_pca8_matern32': 'Matérn 3/2',
    'base_cyclic_xproc_pca8_matern52': 'Matérn 5/2',
    'base_cyclic_xproc_pca8_rbf_iso': 'RBF',
}
plot_metrics = angle_metrics.loc[
    angle_metrics['angle_view'].eq('molecular_axis') & angle_metrics['model'].isin(model_labels)
].copy()
plot_metrics['angle_bin'] = pd.Categorical(plot_metrics['angle_bin'], bin_order, ordered=True)
for model, label in model_labels.items():
    line = plot_metrics.loc[plot_metrics['model'].eq(model)].sort_values('angle_bin')
    axes[2].plot(bin_order, line['RMSE'], marker='o', label=label)
axes[2].set(xlabel='Molecular-axis azimuth bin (degree)', ylabel='OOF RMSE', title='Best method changes in the 50–70° regime')
axes[2].tick_params(axis='x', rotation=25)
axes[2].legend()
plt.tight_layout(); plt.show()

## E. 候補構造系列を丸ごとhold outする補助評価

file_keyの最後のtokenを除いたprefixを候補series IDとみなし、同一prefixを訓練・テストへ分けないGroupKFoldも確認します。これは命名規則から推定したgroupなので、物理的意味の確認が必要です。指定固定foldは受領RFとの対応比較、こちらは未知系列外挿という別の問いです。RのRFはこのgroup splitで未実行なので、ここではGPR同士だけを比較します。

In [ ]:
from chemistory_gpr.group_validation import run_prefix_group_comparison

RUN_PREFIX_GROUP_CV = False  # Trueで非ARD 6カーネルを再fit（数十秒程度）
if RUN_PREFIX_GROUP_CV:
    group_configs = [config for config in handoff_kernel_candidates(rbf_ard_restarts=0) if not config.ard]
    group_paths = run_prefix_group_comparison(data, group_configs, RESULTS)
    group_metrics = pd.read_csv(group_paths['metrics'])
else:
    group_metrics = pd.read_csv(RESULTS / 'gpr_handoff_group10_prefix_metrics.csv')
display(group_metrics[['model','kernel_family','matern_nu','R2','RMSE','MAE','coverage_95','NLPD']])
print('固定fold Matérn 3/2 R² = 0.933874: 既知系列内の補間')
print(f"prefix-group最良R² = {group_metrics.iloc[0]['R2']:.6f}: 候補系列の外挿")

## F. 物理角度・相互作用GP・mixtureをnested group CVで比較

生角度7列を `sin(分子軸方位), cos(分子軸方位), 面外傾きproxy, 反平行ずれ` の4変数へ整理します。相互作用GPは `k_axis + k_environment + k_axis×k_environment + White` です。この積項は、全体Matérn 3/2の誤差が高角度へ集中し、同じ角度帯でもH3近傍Mg/Oと面外傾きによって応答が枝分かれしたため導入しました。加法項だけなら軸効果と環境効果は独立ですが、積項は軸も環境も似た試料にだけ強い共分散を与えます。これはfunctional ANOVA / tensor-product GPとして、`配向が環境効果を修飾する` 仮説を表します。mixtureでは40/45/50°を候補とし、高角度専門家をMatérn 1/2またはPython RFにします。モデルと境界は外側trajectory groupを見ず、内側group CVのRMSEだけで選びます。

既定は保存済み結果を表示します。再fitはColabで時間を要するため、必要なtoggleだけTrueにしてください。

In [ ]:
from chemistory_gpr.nested_group import default_nested_candidates, run_nested_group_comparison
from chemistory_gpr.next_model_report import build_next_model_report

RUN_NESTED_GROUP_CV = False  # True: 外側5-fold×内側4-fold×12候補
RUN_NEXT_MODEL_REPORT_REFIT = False  # True: fixed10/group10/group感度を再fit
if RUN_NESTED_GROUP_CV:
    nested_paths = run_nested_group_comparison(
        data, default_nested_candidates(), RESULTS,
        group_scheme='trajectory', outer_splits=5, inner_splits=4, seed=123,
    )
if RUN_NEXT_MODEL_REPORT_REFIT:
    next_paths = build_next_model_report(data, RESULTS)

token_diagnostics = pd.read_csv(RESULTS / 'gpr_handoff_file_key_token_diagnostics.csv')
fixed_next = pd.read_csv(RESULTS / 'gpr_handoff_fixed10_next_models_metrics.csv')
group10_next = pd.read_csv(RESULTS / 'gpr_handoff_group10_next_models_metrics.csv')
nested_metrics = pd.read_csv(RESULTS / 'gpr_handoff_nested_group_metrics.csv')
nested_selections = pd.read_csv(RESULTS / 'gpr_handoff_nested_group_selections.csv')
group_sensitivity = pd.read_csv(RESULTS / 'gpr_handoff_group_scheme_model_metrics.csv')
components = pd.read_csv(RESULTS / 'gpr_handoff_interaction_kernel_components.csv')
print('file_key tokenと既知物理特徴の対応（名前はまだ暫定）')
display(token_diagnostics)
print('受領RF対応 fixed10: 既知trajectory内補間')
display(fixed_next[['candidate','R2','RMSE','MAE','coverage_95','NLPD']])
print('同じprefix-group10: trajectory外挿候補')
display(group10_next[['candidate','R2','RMSE','MAE','coverage_95','NLPD']])
print('strict nested group estimate')
display(nested_metrics)
display(nested_selections)

In [ ]:
fixed_angle = pd.read_csv(RESULTS / 'gpr_handoff_fixed10_next_models_angle_metrics.csv')
selected = {
    'compact_axis_global_matern32': 'global M3/2',
    'axis_plus_environment_matern32': 'axis + env',
    'axis_environment_interaction_matern32': 'axis + env + interaction',
    'moe_matern32_rf_gate40': '40° gate + RF',
}
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for candidate, label in selected.items():
    line = fixed_angle.loc[fixed_angle['candidate'].eq(candidate)].copy()
    line['angle_bin'] = pd.Categorical(line['angle_bin'], bin_order, ordered=True)
    line = line.sort_values('angle_bin')
    axes[0].plot(bin_order, line['RMSE'], marker='o', label=label)
axes[0].set(xlabel='axis azimuth bin', ylabel='fixed10 RMSE', title='Interaction improves every angle band')
axes[0].tick_params(axis='x', rotation=25); axes[0].legend()
pivot = group_sensitivity.pivot(index='group_scheme', columns='candidate', values='R2')
pivot.plot(kind='bar', ax=axes[1])
axes[1].set(ylabel='OOF R²', title='Result depends on what is held unknown')
axes[1].tick_params(axis='x', rotation=25); axes[1].legend(fontsize=8)
variance_cols = ['axis_additive_variance','environment_additive_variance','interaction_variance']
axes[2].boxplot([components[c] for c in variance_cols], labels=['axis','environment','axis×environment'])
axes[2].set(ylabel='optimized signal variance', title='Product term carries the fitted signal')
plt.tight_layout(); plt.show()

## G. 元3D構造へ戻れる場合

`geometry3d.py` は `file_key, atom_label, element, x, y, z` のlong表から、Mg/Oの分子軸方向への符号付き射影、軸からの垂直距離、H3/H6最短距離・近傍逆距離和、H3−H6非対称性を生成します。少なくともC3/H3/C6/H6と同じ9-cellのMg/O周期像が必要です。

現在のhandoff ZIPには170構造の元座標がなく、dist_autoの座標は別の330例なので、3D新特徴は今回の性能表へ混ぜていません。

In [ ]:
from chemistory_gpr.geometry3d import (
    REQUIRED_COORDINATE_COLUMNS, derive_rotation_invariant_features,
)
print('required raw-coordinate columns =', sorted(REQUIRED_COORDINATE_COLUMNS))
print('raw coordinates are not included; function is ready for a matching 170-file_key table')

## H. 動かせる分子軸模式図・3D fitting面・GP分散

### H.1 trajectoryとOOF予測区間を同期再生

左側は受領summaryから復元できるC3→H3・C6→H6の**方向ベクトル模式図**です。元原子座標ではありません。z符号は不明なのでC6→H6側を正とする鏡映同値な表示を使い、Mg/O位置は捏造しません。右側は全170例のOOF予測、選択trajectory、現在試料の予測平均と95%区間です。Playまたはsliderでtoken 5の進行と予測挙動を同期して確認できます。

In [ ]:
from chemistory_gpr.visualization import (
    fit_full_interaction_gp, interaction_surface_figure, interaction_surface_table,
    molecular_axis_uncertainty_animation, oof_uncertainty_figure, raw_structure_figure,
)

interaction_predictions = pd.read_csv(RESULTS / 'gpr_handoff_fixed10_next_models_predictions.csv')
TRAJECTORY_TO_VIEW = '0-0-3-18'  # 例: '0-0-2-16'; Noneなら全170例
trajectory_figure = molecular_axis_uncertainty_animation(
    data.base, interaction_predictions, trajectory=TRAJECTORY_TO_VIEW,
)
trajectory_figure.show()

### H.2 予測平均・標準偏差・95%区間幅の3D slice

高次元の相互作用GPを、(a) 分子軸方位×面外傾き、(b) 分子軸方位×H3近傍逆距離和の2面で切ります。表示しないbaseとX_procは観測試料 `0-0-3-18-10` に固定しています。面はマウスで回転・拡大でき、上のボタンで平均、標準偏差、95%幅、上下限を切り替えられます。これは全データfitの説明用条件付き断面であり、元3D原子構造でもOOF評価面でもありません。分散が大きい領域の平均値は、物理傾向として強く解釈しないでください。

In [ ]:
axis_tilt_surface = pd.read_csv(RESULTS / 'gpr_handoff_interaction_surface_axis_tilt.csv')
h3_environment_surface = pd.read_csv(RESULTS / 'gpr_handoff_interaction_surface_h3_environment.csv')
print('方位 × 面外傾きproxy')
interaction_surface_figure(axis_tilt_surface, data.base).show()
print('方位 × H3近傍逆距離和')
interaction_surface_figure(h3_environment_surface, data.base).show()

In [ ]:
# 別の基準試料・環境特徴で条件付き面を再fitする場合だけTrue
RUN_CUSTOM_SURFACE = False
CUSTOM_REFERENCE_FILE_KEY = '0-0-3-18-10'
CUSTOM_SURFACE_FEATURE = 'sum_invd_LH3_d5'  # axis_tilt_deg, antiparallel_deviation_deg, またはbase列
if RUN_CUSTOM_SURFACE:
    full_interaction_model = fit_full_interaction_gp(data, seed=123)
    custom_surface = interaction_surface_table(
        data, model=full_interaction_model,
        reference_file_key=CUSTOM_REFERENCE_FILE_KEY,
        surface_feature=CUSTOM_SURFACE_FEATURE,
    )
    interaction_surface_figure(custom_surface, data.base).show()

### H.3 OOF不確実性と実座標viewer

左図の縦線は各試料の95%予測区間、右図は予測標準偏差と絶対誤差です。95% coverageは0.976ですが、標準偏差と絶対誤差の順位相関は約0.199なので、分散は完全な誤差検出器ではありません。対応するlong形式の元座標CSVが得られた後は、下の `RAW_COORDINATE_CSV` を指定するとC/H/Mg/Oを実座標で回転表示できます。

In [ ]:
oof_uncertainty_figure(data.base, interaction_predictions).show()

RAW_COORDINATE_CSV = None  # 例: PROJECT_ROOT / 'data' / 'gpr_handoff' / 'raw_coordinates.csv'
RAW_FILE_KEY = '0-0-3-18-10'
if RAW_COORDINATE_CSV is not None:
    raw_coordinates = pd.read_csv(RAW_COORDINATE_CSV)
    raw_structure_figure(raw_coordinates, RAW_FILE_KEY).show()
else:
    print('元座標は未受領です。現在のtrajectory図は方向ベクトル模式図です。')

### 読み方

相互作用GPは固定foldでR²=0.9738、RMSE=1.639となり、全体Matérn 3/2とRFを上回ります。改善は全角度帯で見られ、最適化でも加法項より軸×環境の積項が主に使われます。一方、trajectoryを丸ごと未知にする同じgroup10ではR²=0.365、strict nestedでは0.334です。したがって、既知trajectory内補間では相互作用GPが新しい主候補ですが、未知系列外挿は未解決です。mixtureは高角度帯だけなら有利でも、境界・専門家の内側選択が外側foldで安定しません。file_keyの正式なtoken定義を確認して外側groupを固定し、次の独立系列では相互作用GPを事前指定して評価してください。